# Demo tiền xử lý: hai raw → hai clean

Quan sát phạm vi, cột, giá trị 0, định dạng ngày và kiểm tra chất lượng. Các bảng trung gian dùng để giải thích; CSV cuối do chính hàm xử lý trong source tạo.

Chạy từ trên xuống bằng **Restart Kernel → Run All**, hoặc Shift+Enter từng cell. Thuật toán dùng lại từ `source/`; không sửa các CSV nghiên cứu đã có. Xem `notebooks/README.md` để cấu hình đường dẫn/kernel.

## Đường dẫn và môi trường
Kiểm tra các đường dẫn trước khi chạy. Các notebook độc lập: mặc định đọc clean/preparation đã lưu; không yêu cầu chạy notebook trước để có biến trong RAM.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                     if (p/'source/preprocessing').is_dir() and (p/'notebooks').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or notebooks directory.')
for folder in [PROJECT_ROOT, PROJECT_ROOT/'notebooks', PROJECT_ROOT/'source/preprocessing']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
from demo_support import (configure, show_paths, demo_directory, save_table, save_json,
                          show_source, show_png, environment, compare_values, read_series, series_plot)
PATHS = configure(PROJECT_ROOT)
# Nếu cần, sửa PATHS.population_raw / history_raw / population_clean / history_clean / preparation tại đây.
show_paths(PATHS)
print('Python kernel:', sys.executable)


,setting,path,exists
0,project,D:\Phân tích kinh doanh_2627\GitHub\IS403.R14....,True
1,data_root,D:\Phân tích kinh doanh_2627,True
2,population_raw,D:\Phân tích kinh doanh_2627\archive\Electric_...,True
3,history_raw,D:\Phân tích kinh doanh_2627\Electric_Vehicle_...,True
4,population_clean,D:\Phân tích kinh doanh_2627\archive\EV_Popula...,True
5,history_clean,D:\Phân tích kinh doanh_2627\Electric_Vehicle_...,True
6,preparation,D:\Phân tích kinh doanh_2627\Du_lieu_dan_xuat\...,True
7,output_root,D:\Phân tích kinh doanh_2627\Du_lieu_dan_xuat\...,True


Python kernel: C:\Users\ASUS\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
OUT = demo_directory(PATHS, 'preprocessing')
print('Output demo:', OUT)
display(environment())


Output demo: D:\Phân tích kinh doanh_2627\Du_lieu_dan_xuat\notebook_demo\preprocessing_20261116_035834_d4633229


,package,version,status
0,pandas,3.0.5,Đã tìm thấy
1,numpy,2.5.3,Đã tìm thấy
2,scipy,1.18.1,Đã tìm thấy
3,matplotlib,3.11.2,Đã tìm thấy
4,statsmodels,0.15.0,Đã tìm thấy
5,scikit-learn,1.9.1,Đã tìm thấy
6,xgboost,3.4.1,Đã tìm thấy
7,ipykernel,7.3.0,Đã tìm thấy


## Đọc raw

In [3]:
from source.preprocessing import clean_population as cp, clean_history as ch
from source.preprocessing.run_preprocessing import check_data
population_raw = pd.read_csv(PATHS.population_raw, low_memory=False)
history_raw = pd.read_csv(PATHS.history_raw, low_memory=False)
display(pd.DataFrame({'dataset':['Population raw','History raw'],
                      'rows':[len(population_raw),len(history_raw)],
                      'columns':[len(population_raw.columns),len(history_raw.columns)]}))
display(population_raw[['County','City','State','Model Year','Electric Range']].head())
display(history_raw[['Date','County','State','Vehicle Primary Use']].head())


,dataset,rows,columns
0,Population raw,135038,17
1,History raw,19700,10


,County,City,State,Model Year,Electric Range
0,Thurston,Tumwater,WA,2019,220.0
1,Island,Clinton,WA,2022,0.0
2,Snohomish,Snohomish,WA,2020,266.0
3,King,Seattle,WA,2020,322.0
4,Snohomish,Edmonds,WA,2013,208.0


,Date,County,State,Vehicle Primary Use
0,October 31 2019,Flathead,MT,Passenger
1,October 31 2019,New London,CT,Passenger
2,June 30 2019,Pend Oreille,WA,Truck
3,November 30 2019,Virginia Beach,VA,Passenger
4,February 28 2018,Sacramento,CA,Passenger


## Phạm vi WA và lựa chọn cột
Lọc phạm vi không phải xóa trùng. Đây là view quan sát trước khi gọi hàm tạo clean.

In [4]:
scope = []
for label, raw in [('Population',population_raw),('History',history_raw)]:
    keep = raw['State'].eq('WA')
    scope.append({'dataset':label,'raw_rows':len(raw),'WA_rows':int(keep.sum()),
                  'excluded_rows':int((~keep).sum()),'missing_state':int(raw['State'].isna().sum())})
display(pd.DataFrame(scope))
display(pd.DataFrame({'retained_column':cp.COLUMNS}))
print('Population columns removed:', [x for x in population_raw.columns if x not in cp.COLUMNS])


,dataset,raw_rows,WA_rows,excluded_rows,missing_state
0,Population,135038,134726,312,0
1,History,19700,6474,13226,83


,retained_column
0,County
1,City
2,State
3,Postal Code
4,Model Year
5,Make
6,Model
7,Electric Vehicle Type
8,Clean Alternative Fuel Vehicle (CAFV) Eligibility
9,Electric Range


Population columns removed: ['VIN (1-10)', 'Legislative District', 'DOL Vehicle ID', 'Vehicle Location', 'Electric Utility', '2020 Census Tract']


## Quy tắc xử lý trong source
Xem chính hàm sẽ chạy: WA, giữ cột, 0→thiếu, kiểu số và stable sort. Không có bước tự xóa các dòng giống nhau trên thuộc tính giữ lại.

In [5]:
show_source(cp.clean_population)
show_source(ch.clean_history)


def clean_population(raw_path: Path) -> tuple[pd.DataFrame, dict]:
    raw = pd.read_csv(raw_path, low_memory=False)
    missing = set(COLUMNS) - set(raw.columns)
    if missing:
        raise ValueError(f'Missing source columns: {sorted(missing)}')
    clean = raw.loc[raw['State'].eq('WA'), COLUMNS].copy()
    zeros = {col: int(clean[col].eq(0).sum()) for col in ['Electric Range', 'Base MSRP']}
    for col in zeros:
        clean[col] = clean[col].mask(clean[col].eq(0))
    # Use a consistent numeric representation when exporting CSV.
    # Integer-valued attributes can be recast in subsequent analytical views.
    for col in NUMERIC:
        clean[col] = pd.to_numeric(clean[col], errors='raise').astype('float64')
    clean = clean.sort_values(SORT_KEYS, kind='stable', na_position='last').reset_index(drop=True)
    audit = {
        'raw_shape': list(raw.shape), 'clean_shape': list(clean.shape),
        'removed_rows_non_WA': int(len(raw) - len(clean)),
        'removed_columns': [col for col in raw if col not in COLUMNS],
        'zeros_converted_to_missing': zeros,
        'missing_after': {col: int(n) for col, n in clean.isna().sum().items()},
        'duplicates_on_original_all_columns': int(raw.duplicated().sum()),
        'repeated_rows_on_retained_columns_preserved': int(clean.duplicated().sum()),
        'sort_keys': SORT_KEYS, 'sort_ties': 'preserve source order',
    }
    return clean, audit

def clean_history(raw_path: Path) -> tuple[pd.DataFrame, dict]:
    raw = pd.read_csv(raw_path, low_memory=False)
    if list(raw.columns) != COLUMNS:
        raise ValueError('History source columns do not match the expected schema.')
    clean = raw.loc[raw['State'].eq('WA'), COLUMNS].copy()
    clean['Date'] = pd.to_datetime(clean['Date'], format='%B %d %Y', errors='raise').dt.strftime('%Y-%m-%d')
    for col in COLUMNS[4:]:
        clean[col] = pd.to_numeric(clean[col], errors='raise').astype('float64')
    # Preserve source order for use groups within the same date and county.
    clean = clean.sort_values(['Date', 'County'], kind='stable').reset_index(drop=True)
    audit = {
        'raw_shape': list(raw.shape), 'clean_shape': list(clean.shape),
        'removed_rows_non_WA_or_missing_state': int(len(raw) - len(clean)),
        'raw_missing_state': int(raw['State'].isna().sum()),
        'missing_after': {col: int(n) for col, n in clean.isna().sum().items()},
        'dates': int(clean.Date.nunique()), 'counties': int(clean.County.nunique()),
        'uses': clean['Vehicle Primary Use'].value_counts().to_dict(),
        'first_date': clean.Date.min(), 'last_date': clean.Date.max(),
        'sort_keys': ['Date', 'County'], 'sort_ties': 'preserve source order',
    }
    return clean, audit

## Chạy xử lý Population
Không điền thiếu bằng trung bình hoặc xóa phương tiện vì Range thiếu.

In [6]:
population, population_audit = cp.clean_population(PATHS.population_raw)
wa = population_raw.loc[population_raw['State'].eq('WA')]
display(pd.DataFrame([{'column':col,'zeros_before':int(wa[col].eq(0).sum()),
                       'missing_before':int(wa[col].isna().sum()),
                       'zeros_after':int(population[col].eq(0).sum()),
                       'missing_after':int(population[col].isna().sum())}
                      for col in ['Electric Range','Base MSRP']]))
display(population.head())
display(population.dtypes.rename('dtype').to_frame())
print('Repeated retained rows preserved:', population_audit['repeated_rows_on_retained_columns_preserved'])


,column,zeros_before,missing_before,zeros_after,missing_after
0,Electric Range,56868,1,0,56869
1,Base MSRP,131306,1,0,131307


,County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Base MSRP
0,Adams,Hatton,WA,99344.0,2020.0,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,266.0,NaN
1,Adams,Lind,WA,99341.0,2020.0,CHRYSLER,PACIFICA,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,32.0,NaN
2,Adams,Lind,WA,99341.0,2022.0,RIVIAN,R1T,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,NaN,NaN
3,Adams,Othello,WA,99344.0,2011.0,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,73.0,NaN
4,Adams,Othello,WA,99344.0,2012.0,CHEVROLET,VOLT,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,35.0,NaN


,dtype
County,str
City,str
State,str
Postal Code,float64
Model Year,float64
Make,str
Model,str
Electric Vehicle Type,str
Clean Alternative Fuel Vehicle (CAFV) Eligibility,str
Electric Range,float64


Repeated retained rows preserved: 93707


## Chạy xử lý History
Vẫn giữ cấp tháng–quận–nhóm sử dụng, chưa cộng thành chuỗi toàn bang.

In [7]:
history, history_audit = ch.clean_history(PATHS.history_raw)
display(pd.DataFrame({'raw_date':history_raw.loc[history_raw['State'].eq('WA'),'Date'].head().tolist()}))
display(history[['Date','County','Vehicle Primary Use']].head())
display(pd.DataFrame([{'months':history.Date.nunique(),'counties':history.County.nunique(),
                       'use_groups':history['Vehicle Primary Use'].nunique(),'rows':len(history)}]))


,raw_date
0,June 30 2019
1,April 30 2021
2,May 31 2023
3,November 30 2020
4,February 28 2017


,Date,County,Vehicle Primary Use
0,2017-01-31,Adams,Passenger
1,2017-01-31,Adams,Truck
2,2017-01-31,Asotin,Passenger
3,2017-01-31,Asotin,Truck
4,2017-01-31,Benton,Passenger


,months,counties,use_groups,rows
0,83,39,2,6474


## Kiểm tra và đối chiếu trước–sau
Kiểm tra chất lượng không phải một phép biến đổi raw mới. Không dùng clean có sẵn để tạo đầu ra.

In [8]:
quality = check_data(population, history, population_audit, history_audit)
display(pd.DataFrame([quality['quality']]).T.rename(columns={0:'result'}))
display(pd.DataFrame([{'dataset':'Population','raw_shape':population_raw.shape,'clean_shape':population.shape},
                      {'dataset':'History','raw_shape':history_raw.shape,'clean_shape':history.shape}]))
pd.testing.assert_frame_equal(population, pd.read_csv(PATHS.population_clean), check_dtype=False)
pd.testing.assert_frame_equal(history, pd.read_csv(PATHS.history_clean), check_dtype=False)



,result
WA_only,True
population_row_accounting_valid,True
population_range_msrp_zeros,0
history_missing_cells,0
history_duplicate_keys,0
history_complete_combinations,6474
history_consecutive_months,83
history_count_identities_valid,True
history_percentages_valid,True


,dataset,raw_shape,clean_shape
0,Population,"(135038, 17)","(134726, 11)"
1,History,"(19700, 10)","(6474, 10)"


## Xuất bản demo riêng
Chỉ xuất vào OUT mới, không ghi đè clean hoặc results chính thức.

In [9]:
for frame,name in [(population,'EV_Population_Washington_Clean.csv'),(history,'EV_Population_History_Washington_Clean.csv')]:
    if (OUT/name).exists(): raise FileExistsError(OUT/name)
    cp.write_csv(frame, OUT/name)
save_json(quality, OUT/'data_quality_summary.json')
display(pd.DataFrame({'file':[p.name for p in OUT.iterdir()]}))


,file
0,data_quality_summary.json
1,EV_Population_History_Washington_Clean.csv
2,EV_Population_Washington_Clean.csv
